# 제품별 단일 프롬프트 생성 (옵션 B, v3: product_info (1).json 사용)

In [ ]:
# =============================
# 0) CONFIG
# =============================
from pathlib import Path

PERSONA_JSONL = Path("../people_segment/persona_attributes_weighted.jsonl")
PRODUCT_JSON  = Path("../product_info/product_info_preprocessed.json")
OUT_JSONL     = Path("prompts_B.jsonl")
OUT_PREVIEW   = Path("prompts_B_preview.json")

LIMIT_PRODUCTS = None
LIMIT_PERSONAS = None
print("CONFIG loaded.")

CONFIG loaded.


In [3]:
# =============================
# 1) Load data
# =============================
import json

# Personas
personas = []
with open(PERSONA_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            personas.append(json.loads(line))

# Products from JSON
products_raw = json.loads(PRODUCT_JSON.read_text(encoding="utf-8"))

def as_list(val):
    if val is None: return []
    parts = [p.strip() for p in str(val).replace(";", ",").split(",")]
    return [p for p in parts if p]

def price_text(v):
    try:
        iv = int(v)
        return f"{iv:,}원"
    except Exception:
        return str(v) if v is not None else None

def to_product_dict(r):
    cat = r.get("category") or {}
    features = as_list(r.get("feature"))
    targeted = as_list(r.get("targeted_consumer"))
    return {
        "product_id": r.get("product_id") or r.get("id") or "",
        "product_name": r.get("product_name"),
        "category": cat.get("level_1"),
        "subcategory": " > ".join([c for c in [cat.get("level_2"), cat.get("level_3")] if c]),
        "features": features,
        "targeted_consumer": targeted,
        "launch_ym": r.get("launch_ym"),
        "price": price_text(r.get("price")),
        "ad_model": r.get("ad_model"),
        "advertise": r.get("advertise"),
    }

def product_block(p):
    lines = [
        f"- product_id: {p.get('product_id','')}",
        f"- 제품명: {p.get('product_name','')}"
    ]
    if p.get("category"): lines.append(f"- 카테고리: {p['category']}")
    if p.get("subcategory"): lines.append(f"- 서브카테고리: {p['subcategory']}")
    if p.get("features"): lines.append(f"- 주요 특징: {', '.join(p['features'])}")
    if p.get("targeted_consumer"): lines.append(f"- 타깃: {', '.join(p['targeted_consumer'])}")
    if p.get("price"): lines.append(f"- 기준 가격대: {p['price']}")
    if p.get("advertise"): lines.append(f"- 광고/프로모션: {p['advertise']}")
    return "\n".join(lines)

products = [to_product_dict(r) for r in products_raw]
if LIMIT_PRODUCTS: products = products[:LIMIT_PRODUCTS]
if LIMIT_PERSONAS: personas = personas[:LIMIT_PERSONAS]
print("Loaded:", len(personas), "personas /", len(products), "products")

Loaded: 363 personas / 15 products


In [4]:
# =============================
# 2) Helpers
# =============================
from typing import Dict, Any

def format_attributes_for_prompt(attrs: Dict[str, Any]) -> str:
    lines = []
    for k, vw in attrs.items():
        v = vw.get("value", None)
        w = vw.get("weight", 0.0)
        v_str = "None" if v is None else str(v)
        lines.append(f"- {k}: {v_str} (w={w:.3f})")
    return "\n".join(lines[:60])

def build_single_prompt(product: Dict[str, Any], persona: Dict[str, Any]) -> str:
    meta = persona.get("meta", {}) or {}
    cluster = meta.get("cluster", "")
    label = meta.get("label", "")
    desc = meta.get("desc", meta.get("Description",""))
    cluster_block = f"""
[클러스터 컨텍스트]
- cluster: {cluster}
- label: {label}
- desc: {desc}
""".strip() if (cluster or label or desc) else "[클러스터 컨텍스트]\n- N/A"

    return f"""
[역할]
당신은 한국 소비자 데이터 분석가이자 마케팅 전문가입니다.
아래의 "제품 정보"와 "페르소나"를 바탕으로,
이 페르소나가 해당 제품의 구매자로서 성립하는 **싱글 턴** 페르소나 JSON을 생성하세요.

[제품 정보]
{product_block(product)}

[페르소나]
- id: {persona.get('persona_key','')}
- 속성(가중치 합=1):
{format_attributes_for_prompt(persona.get('attributes', {}))}

{cluster_block}

[규칙]
- '클러스터 컨텍스트'는 배경 지침으로만 사용합니다. 속성 가중치(합=1)와 충돌 시 '속성 가중치'를 우선합니다.
- 2024-07 ~ 2025-06 월별로 구매확률(prob 0~1)과 예상수량(qty 정수)을 제시합니다.
- 추석/설, 광고/프로모션/계절성을 반영합니다.
- **반드시 아래 JSON 스키마를 출력**하고, 불필요한 설명 문장은 출력하지 마세요.

[출력 스키마(JSON)]
{{
  "persona_id": "p_{{product_id}}_{persona.get('persona_key','')}",
  "product_id": "{{product_id}}",
  "segment_ref": "{persona.get('persona_key','')}",
  "attributes": {{ "{{속성명}}": {{"value": "<값>", "weight": <0~1> }}, "...": "..." }},
  "purchase_pattern": {{
    "avg_purchase_prob": <0~1>,
    "avg_purchase_qty": <int>,
    "seasonality": {{"추석": "+x%", "설": "+y%"}},
    "promotion_effect": "광고/프로모션 노출 시 +z%"
  }},
  "forecast_12mo": {{
    "2024-07": {{"prob": <0~1>, "qty": <int>}},
    "...": {{}}, 
    "2025-06": {{"prob": <0~1>, "qty": <int>}}
  }}
}}
""".strip()

In [5]:
# =============================
# 3) Build & save
# =============================
import json
from pathlib import Path

records = []
for prod in products:
    for persona in personas:
        rec = {
            "product": prod,
            "persona": {"persona_key": persona.get("persona_key")},
            "prompt": build_single_prompt(prod, persona)
        }
        records.append(rec)

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

Path(OUT_PREVIEW).write_text(json.dumps(records[:3], ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", OUT_JSONL, "size=", Path(OUT_JSONL).stat().st_size, "bytes")
records[:1]

Saved: prompts_B.jsonl size= 17156040 bytes


[{'product': {'product_id': '',
   'product_name': '덴마크 하이그릭요거트 400g',
   'category': '우유류',
   'subcategory': '발효유 > 호상-중대용량',
   'features': ['건강식품', '고단백', '고소한맛', '높은 만족도'],
   'targeted_consumer': ['유당불내증'],
   'launch_ym': None,
   'price': '3,980원',
   'ad_model': None,
   'advertise': '2025년 6-7월 일반인 광고, SNS 바이럴'},
  'persona': {'persona_key': 1},
  'prompt': '[역할]\n당신은 한국 소비자 데이터 분석가이자 마케팅 전문가입니다.\n아래의 "제품 정보"와 "페르소나"를 바탕으로,\n이 페르소나가 해당 제품의 구매자로서 성립하는 **싱글 턴** 페르소나 JSON을 생성하세요.\n\n[제품 정보]\n- product_id: \n- 제품명: 덴마크 하이그릭요거트 400g\n- 카테고리: 우유류\n- 서브카테고리: 발효유 > 호상-중대용량\n- 주요 특징: 건강식품, 고단백, 고소한맛, 높은 만족도\n- 타깃: 유당불내증\n- 기준 가격대: 3,980원\n- 광고/프로모션: 2025년 6-7월 일반인 광고, SNS 바이럴\n\n[페르소나]\n- id: 1\n- 속성(가중치 합=1):\n- gender: 남자 (w=0.043)\n- age: 30대 (w=0.043)\n- job: 사무 종사자 (w=0.043)\n- education: 대학교 졸업(전문대졸/대학원생 포함) (w=0.043)\n- region: 경상북도 (w=0.043)\n- household: 1세대가족 (w=0.043)\n- marriage: 기혼 (w=0.043)\n- brand_loyalty_scaled: 0.208955223880597 (w=0.050)\n- cooking_convenience_scale